# Retrain channel-invariant, multihead InstanSeg on CPDMI at 0.325 µm/pixel

This is a separate architectural-correction notebook based on `instanseg_train_cpdmi_0325.ipynb`. It retains the documented CPDMI training recipe at 0.325 µm/pixel but explicitly sets `multihead=True`, matching the two-decoder structure found in the released `fluorescence_nuclei_and_cells` model.

The notebook performs three checks before a long run:

1. Build the trainer's model directly and require two independent decoders.
2. Run channel-invariant forward passes with 1, 2, and 8 input channels and require a 10-channel output.
3. Optionally run a short training smoke test through both hot-start and main-loss phases, then reload its checkpoint and verify the two-decoder architecture.

The saved combined dataset is reused and filtered to CPDMI Vectra + Zeiss records in memory. TissueNet and CODEX remain excluded. Full training is disabled by default.

In [17]:
%load_ext autoreload
%autoreload 2

import os
import shlex
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import instanseg

TRAINING_ROOT = Path(
    os.environ.get("INSTANSEG_TRAINING_ROOT", "/data1/lowes/ratnayn/Data/instanseg")
).expanduser().resolve()
FORK_ROOT = Path(
    os.environ.get("INSTANSEG_FORK_ROOT", "/data1/lowes/ratnayn/Codex/projects/instanseg")
).expanduser().resolve()
DATASET_PATH = TRAINING_ROOT / "datasets"
MODEL_PATH = TRAINING_ROOT / "models"
COMBINED_DATASET_FILE = DATASET_PATH / "segmentation_dataset.pth"

instanseg_import_path = Path(instanseg.__file__).resolve()
print(f"InstanSeg imported from: {instanseg_import_path}")
assert instanseg_import_path.is_relative_to(FORK_ROOT), (
    f"Expected the editable InstanSeg fork under {FORK_ROOT}, but imported {instanseg_import_path}. "
    "Restart the kernel with the instanseg_training environment after installing the fork."
)
assert COMBINED_DATASET_FILE.exists(), f"Missing saved dataset: {COMBINED_DATASET_FILE}"
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(torch.cuda.current_device())
    print(f"GPU: {properties.name} ({properties.total_memory / 1024**3:.1f} GiB)")
else:
    print("CUDA is unavailable. Architecture checks can run on CPU; training cells require CUDA.")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
InstanSeg imported from: /data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/__init__.py
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA H200 (139.8 GiB)


## Load the saved dataset and isolate CPDMI

`torch.load(..., weights_only=False)` is used because this is our trusted dataset checkpoint containing arrays and metadata. No downloader or dataset rebuild is needed.

In [18]:
Combined_Dataset = torch.load(
    COMBINED_DATASET_FILE,
    map_location="cpu",
    weights_only=False,
)

CPDMI_Dataset = {
    split: [item for item in items if item.get("parent_dataset") == "CPDMI_2023"]
    for split, items in Combined_Dataset.items()
}
del Combined_Dataset

assert CPDMI_Dataset["Train"], "No CPDMI training records found"
assert CPDMI_Dataset["Validation"], "No CPDMI validation records found"
assert all(
    item.get("parent_dataset") == "CPDMI_2023"
    for items in CPDMI_Dataset.values()
    for item in items
)
print({split: len(items) for split, items in CPDMI_Dataset.items()})

{'Train': 116, 'Validation': 30, 'Test': 0}


In [19]:
for split in ("Train", "Validation", "Test"):
    items = CPDMI_Dataset[split]
    if not items:
        print(f"{split}: empty (CODEX remains excluded)")
        continue
    paired = sum(
        ("nucleus_masks" in item or "masks" in item) and "cell_masks" in item
        for item in items
    )
    print(
        f"{split}: items={len(items)}, paired_NC={paired}, "
        f"platforms={dict(Counter(item.get('platform') for item in items))}, "
        f"pixel_sizes={sorted({float(item['pixel_size']) for item in items})}, "
        f"channels={sorted({len(item.get('channel_names', [])) for item in items})}"
    )

Train: items=116, paired_NC=36, platforms={'Vectra': 101, 'Zeiss': 15}, pixel_sizes=[0.25, 0.325, 0.5], channels=[5, 7, 8]
Validation: items=30, paired_NC=7, platforms={'Vectra': 26, 'Zeiss': 4}, pixel_sizes=[0.25, 0.325, 0.5], channels=[5, 7, 8]
Test: empty (CODEX remains excluded)


## Architecture preflight: require the released-model topology

In this source version, `multihead=False` flattens the nucleus and cell outputs into one shared decoder. `multihead=True` creates two full decoders, each with coordinate, sigma, and seed projections totaling five channels. The channel-invariant adaptor accepts an arbitrary input-channel count and emits the fixed three-channel representation consumed by the two-decoder InstanSeg U-Net.

This cell uses the same `build_model_from_dict()` and `AdaptorNetWrapper` functions used by the trainer. It is intentionally independent of the long training call.

In [20]:
from instanseg.utils.model_loader import build_model_from_dict
from instanseg.utils.models.ChannelInvariantNet import AdaptorNetWrapper, has_AdaptorNet

architecture_config = dict(
    model_str="InstanSeg_UNet",
    dim_in=None,
    dropprob=0.0,
    multihead=True,
    cells_and_nuclei=True,
    dim_coords=2,
    n_sigma=2,
    dim_seeds=1,
    layers=[32, 64, 128, 256],
    norm="BATCH",
)

base_model = build_model_from_dict(architecture_config, random_seed=42)
assert len(base_model.decoders) == 2, (
    f"Expected separate nucleus and cell decoders, found {len(base_model.decoders)}"
)
assert base_model.decoders[0] is not base_model.decoders[1]
assert [len(decoder.final_block) for decoder in base_model.decoders] == [3, 3]

decoder_parameter_counts = [
    sum(parameter.numel() for parameter in decoder.parameters())
    for decoder in base_model.decoders
]
decoder_parameter_ids = [
    {id(parameter) for parameter in decoder.parameters()}
    for decoder in base_model.decoders
]
assert decoder_parameter_ids[0].isdisjoint(decoder_parameter_ids[1])
print(f"Decoder count: {len(base_model.decoders)}")
print(f"Decoder parameter counts: {decoder_parameter_counts}")

model = AdaptorNetWrapper(
    base_model,
    adaptor_net_str="1",
    norm="BATCH",
).eval()
assert has_AdaptorNet(model)

with torch.inference_mode():
    for channel_count in (1, 2, 8):
        example = torch.randn(1, channel_count, 64, 64)
        output = model(example)
        assert output.shape == (1, 10, 64, 64), output.shape
        assert torch.isfinite(output).all()
        print(f"{channel_count} input channel(s) -> {tuple(output.shape)}")

del output, example, model, base_model
print("PASS: channel-invariant two-decoder architecture is operational.")

Generating InstanSeg_UNet
Decoder count: 2
Decoder parameter counts: [1036399, 1036399]
1 input channel(s) -> (1, 10, 64, 64)
2 input channel(s) -> (1, 10, 64, 64)
8 input channel(s) -> (1, 10, 64, 64)
PASS: channel-invariant two-decoder architecture is operational.


## Published-style 0.325-µm configuration with explicit multihead

The controlled configuration uses 256×256 crops and a 128-pixel instance window. At 0.325 µm/pixel these span 83.2 µm and 41.6 µm, respectively. `length_of_epoch=3000` with batch size three produces approximately 1,000 optimizer steps per epoch, matching the paper's reported schedule. Heavy augmentation is retained.

The essential correction is explicit in both the Python arguments and generated CLI command: `multihead=True`.

In [21]:
EXPERIMENT_NAME = "cpdmi_0325_t256_w128_heavy_multihead"
BATCH_SIZE = 3
NUM_WORKERS = 8
NUM_EPOCHS = 500
LENGTH_OF_EPOCH = 3000
TILE_SIZE = 384
WINDOW_SIZE = 128
RNG_SEED = 42

training_kwargs = dict(
    source_dataset="[CPDMI_2023]",
    output_path=str(MODEL_PATH),
    experiment_str=EXPERIMENT_NAME,
    requested_pixel_size=0.325,
    target_segmentation="NC",
    channel_invariant=True,
    multihead=True,
    augmentation_type="heavy",
    weight=False,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    num_epochs=NUM_EPOCHS,
    length_of_epoch=LENGTH_OF_EPOCH,
    lr=0.001,
    tile_size=TILE_SIZE,
    window_size=WINDOW_SIZE,
    mean_object_diameter=None,
    hotstart_training=10,
    rng_seed=RNG_SEED,
)
assert training_kwargs["channel_invariant"] is True
assert training_kwargs["multihead"] is True
training_kwargs

{'source_dataset': '[CPDMI_2023]',
 'output_path': '/data1/lowes/ratnayn/Data/instanseg/models',
 'experiment_str': 'cpdmi_0325_t256_w128_heavy_multihead',
 'requested_pixel_size': 0.325,
 'target_segmentation': 'NC',
 'channel_invariant': True,
 'multihead': True,
 'augmentation_type': 'heavy',
 'weight': False,
 'batch_size': 3,
 'num_workers': 8,
 'num_epochs': 500,
 'length_of_epoch': 3000,
 'lr': 0.001,
 'tile_size': 384,
 'window_size': 128,
 'mean_object_diameter': None,
 'hotstart_training': 10,
 'rng_seed': 42}

## Two-phase smoke test

The smoke test uses five optimizer batches per phase, one hot-start epoch, and one main epoch. This exercises both loss configurations, joint nucleus/cell loss routing through separate decoders, validation postprocessing, and checkpoint serialization. It uses a distinct output directory and refuses to overwrite an existing result. Set the gate to `True` when a CUDA allocation is active.

In [22]:
RUN_SMOKE_TEST = True
SMOKE_EXPERIMENT_NAME = f"{EXPERIMENT_NAME}_smoke"
SMOKE_OUTPUT = MODEL_PATH / SMOKE_EXPERIMENT_NAME

if RUN_SMOKE_TEST:
    if not torch.cuda.is_available():
        raise RuntimeError("The training smoke test requires a CUDA allocation.")
    if SMOKE_OUTPUT.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing smoke output: {SMOKE_OUTPUT}. "
            "Change SMOKE_EXPERIMENT_NAME or move the existing result deliberately."
        )

    from instanseg.scripts.train import instanseg_training

    smoke_kwargs = dict(training_kwargs)
    smoke_kwargs.update(
        experiment_str=SMOKE_EXPERIMENT_NAME,
        num_epochs=1,
        length_of_epoch=BATCH_SIZE * 5,
        hotstart_training=1,
    )
    torch.cuda.reset_peak_memory_stats()
    instanseg_training(segmentation_dataset=CPDMI_Dataset, **smoke_kwargs)
    print(f"Peak allocated CUDA memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")
    print(f"Peak reserved CUDA memory: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GiB")
else:
    print("Smoke test disabled. Set RUN_SMOKE_TEST=True on a CUDA allocation.")

Saving results to /data1/lowes/ratnayn/Data/instanseg/models/cpdmi_0325_t256_w128_heavy_multihead_smoke
Setting RNG seed to 42
Generating InstanSeg_UNet
<class 'list'> ['cpdmi_2023']
Datasets available in  Train
{('CPDMI_2023', 116)}
After filtering using:
{('CPDMI_2023', 116)}
Datasets available in  Validation
{('CPDMI_2023', 30)}
After filtering using:
{('CPDMI_2023', 30)}
Hotstart for 1 epochs with binary_xloss and dice_loss
Epoch: 0


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

/data1/lowes/ratnayn/Codex/projects/instanseg/instanseg/utils/AI_utils.py:96: RuntimeWarning: Mean of empty slice
  mean1_f1 = np.nanmean(f1_array, axis=0)


Saving best model, best f1_score: 0.0
train_loss: 1.6603, test_loss: 1.5482, training_time: 2, testing_time: 44, f1_score_nuclei: nan, f1_score_cells: 0, f1_score_joint: 0
Starting main training loop with l1_distance and lovasz_hinge
Epoch: 0


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Saving best model, best f1_score: 0.0
train_loss: 4.965, test_loss: 4.9388, training_time: 1, testing_time: 3, f1_score_nuclei: 0, f1_score_cells: 0, f1_score_joint: 0
Generating InstanSeg_UNet
Peak allocated CUDA memory: 11.50 GiB
Peak reserved CUDA memory: 39.84 GiB


## Verify the smoke-test artifact

Run this after the smoke test. It verifies both the recorded configuration and the architecture reconstructed from the saved checkpoint.

In [23]:
if RUN_SMOKE_TEST:
    experiment_log = pd.read_csv(SMOKE_OUTPUT / "experiment_log.csv", header=None)
    recorded = dict(zip(experiment_log.iloc[:, 0], experiment_log.iloc[:, 1]))
    assert str(recorded["channel_invariant"]).lower() == "true"
    assert str(recorded["multihead"]).lower() == "true"

    from instanseg.utils.model_loader import load_model

    reloaded_model, reloaded_config = load_model(
        folder=SMOKE_EXPERIMENT_NAME,
        path=MODEL_PATH,
        device="cpu",
    )
    core_model = reloaded_model.module if hasattr(reloaded_model, "module") else reloaded_model
    core_model = core_model.model if hasattr(core_model, "model") else core_model
    assert len(core_model.decoders) == 2
    assert [len(decoder.final_block) for decoder in core_model.decoders] == [3, 3]
    assert bool(reloaded_config["multihead"]) is True
    print("PASS: smoke checkpoint records multihead=True and reloads with two decoders.")
else:
    print("Smoke verification skipped because RUN_SMOKE_TEST=False.")

Generating InstanSeg_UNet
PASS: smoke checkpoint records multihead=True and reloads with two decoders.


## Full training

Keep this disabled until the architecture preflight and smoke-test verification pass. For a long run, prefer the generated CLI command or the repository's SLURM wrapper so stdout/stderr, immutable source, and checkpoint-resume behavior are recorded.

In [ ]:
RUN_FULL_TRAINING = True

if RUN_FULL_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Full training requires a CUDA allocation.")
    full_output = MODEL_PATH / EXPERIMENT_NAME
    if full_output.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {full_output}")
    from instanseg.scripts.train import instanseg_training
    instanseg_training(segmentation_dataset=CPDMI_Dataset, **training_kwargs)
else:
    print("Full training disabled. Use the CLI/SLURM workflow after the smoke test passes.")

In [ ]:
cli_command = [
    sys.executable, "-m", "instanseg.scripts.train",
    "--data_path", str(DATASET_PATH),
    "--dataset", "segmentation",
    "--source_dataset", "[CPDMI_2023]",
    "--output_path", str(MODEL_PATH),
    "--experiment_str", EXPERIMENT_NAME,
    "--requested_pixel_size", "0.325",
    "--target_segmentation", "NC",
    "--channel_invariant", "True",
    "--multihead", "True",
    "--augmentation_type", "heavy",
    "--weight", "False",
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", str(NUM_WORKERS),
    "--num_epochs", str(NUM_EPOCHS),
    "--length_of_epoch", str(LENGTH_OF_EPOCH),
    "--lr", "0.001",
    "--tile_size", str(TILE_SIZE),
    "--window_size", str(WINDOW_SIZE),
    "--hotstart_training", "10",
    "--rng_seed", str(RNG_SEED),
]
assert cli_command[cli_command.index("--multihead") + 1] == "True"
print(shlex.join(cli_command))

## After training

Compare the multihead best checkpoint against the completed shared-decoder 0.325/256 model and the official 0.5-µm pretrained model using a fixed evaluation set. Report paper-style `F1_mu` over IoU 0.5–0.9, `F1_0.5`, and the current trainer metric separately. Do not compare the live randomized-crop training F1 directly with the paper's fixed benchmark.